# Skincare data analysis

This notebook loads the `data/skincare_log.csv` file and demonstrates exploratory analysis using pandas and DuckDB. It contains example queries and simple plots to answer the README's questions and now includes a small cell to load `data/product_results.csv` for merging and visualization, an interactive Plotly visualization, SQL examples that join the product metadata, and cells to save a DuckDB file and export an interactive HTML report.

In [ ]:
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# Load CSV with pandas
df = pd.read_csv('data/skincare_log.csv', parse_dates=['date'])
df


In [ ]:
# Use DuckDB for SQL-style exploration without creating a DB file
con = duckdb.connect()
con.register('skincare_df', df)
# Example: average hydration by product
print(con.execute('SELECT product, ROUND(AVG(hydration),2) AS avg_hydration, COUNT(*) AS n_records FROM skincare_df GROUP BY product ORDER BY avg_hydration DESC').df())


In [ ]:
# Compute irritation rate by product (fraction of entries with irritation >= 3)
q = '''
SELECT
  product,
  COUNT(*) AS total,
  SUM(CASE WHEN irritation >= 3 THEN 1 ELSE 0 END) AS irritation_count,
  ROUND(100.0 * SUM(CASE WHEN irritation >= 3 THEN 1 ELSE 0 END) / COUNT(*),2) AS irritation_pct
FROM skincare_df
GROUP BY product
ORDER BY irritation_pct DESC
'''
print(con.execute(q).df())


In [ ]:
# Simple static visualizations (kept for quick reference)
sns.set(style='whitegrid')
agg = con.execute('SELECT product, ROUND(AVG(hydration),2) AS avg_hydration, ROUND(AVG(texture),2) AS avg_texture FROM skincare_df GROUP BY product').df()
plt.figure(figsize=(8,4))
sns.barplot(data=agg, x='avg_hydration', y='product', palette='viridis')
plt.title('Average hydration by product')
plt.xlabel('Average hydration')
plt.tight_layout()
plt.show()

# Irritation percent plot
irrit = con.execute('''SELECT product, ROUND(100.0 * SUM(CASE WHEN irritation >= 3 THEN 1 ELSE 0 END) / COUNT(*),2) AS irritation_pct FROM skincare_df GROUP BY product ORDER BY irritation_pct DESC''').df()
plt.figure(figsize=(8,4))
sns.barplot(data=irrit, x='irritation_pct', y='product', palette='magma')
plt.xlabel('Percent of entries with irritation >= 3')
plt.title('Irritation rate by product')
plt.tight_layout()
plt.show()


In [ ]:
# NEW CELL: Load product_results.csv and merge with the skincare log for combined views
prod = pd.read_csv('data/product_results.csv')
# Display the product reference table
display(prod)

# Merge on product name to bring concentration/formulation into the usage log
merged = df.merge(prod[['product','concentration','formulation','before_date','after_date']], on='product', how='left')
display(merged.head())

# Interactive Plotly plot: average hydration by product (merged with product metadata)
agg2 = merged.groupby(['product','formulation'], dropna=False).agg(avg_hydration=('hydration','mean'), n=('hydration','size')).reset_index()
# Use Plotly for interactive bars with hover info
fig = px.bar(agg2.sort_values('avg_hydration', ascending=False), x='avg_hydration', y='product', color='formulation',
             hover_data=['n','formulation'], orientation='h',
             labels={'avg_hydration':'Average hydration','product':'Product'})
fig.update_layout(title='Average hydration by product (merged with product metadata)', yaxis={'categoryorder':'total ascending'}, height=400)
fig.show()


In [ ]:
# NEW CELL: SQL example that joins product metadata into DuckDB and shows combined metrics
# Register the product table in DuckDB as well
con.register('prod_df', prod)
sql = '''
SELECT
  s.product,
  p.concentration,
  p.formulation,
  COUNT(*) AS uses,
  ROUND(AVG(s.hydration),2) AS avg_hydration,
  ROUND(AVG(s.texture),2) AS avg_texture,
  ROUND(100.0 * SUM(CASE WHEN s.irritation >= 3 THEN 1 ELSE 0 END) / COUNT(*),2) AS irritation_pct
FROM skincare_df s
LEFT JOIN prod_df p USING(product)
GROUP BY s.product, p.concentration, p.formulation
ORDER BY avg_hydration DESC
LIMIT 50
'''
print(con.execute(sql).df())


In [ ]:
# SAVE: write merged table to a DuckDB file for downstream analysis
import duckdb
import os

os.makedirs('data', exist_ok=True)
duck = duckdb.connect('data/skincare.duckdb')
# register the pandas DataFrame and create/replace a table in the DuckDB file
duck.register('merged_df', merged)
duck.execute("CREATE OR REPLACE TABLE skincare_merged AS SELECT * FROM merged_df")
duck.close()
print('Wrote data/skincare.duckdb with table `skincare_merged`')


In [ ]:
# EXPORT: save the last Plotly figure to an HTML report for sharing
import os
os.makedirs('reports', exist_ok=True)

# if `fig` exists from the Plotly cell above, write it to HTML
fig.write_html('reports/avg_hydration_by_product.html', include_plotlyjs='cdn')
print('Wrote reports/avg_hydration_by_product.html')


In [ ]:
# BEFORE/AFTER: display side-by-side images for products (requires pillow)
from IPython.display import display
from PIL import Image
import os

def show_before_after(product, before_fname='before.jpg', after_fname='after.jpg'):
    base = os.path.join('data','images', product)
    before_path = os.path.join(base, before_fname)
    after_path = os.path.join(base, after_fname)
    imgs = []
    for p in (before_path, after_path):
        if os.path.exists(p):
            imgs.append(Image.open(p))
        else:
            print(f'Missing: {p}')
            return
    # display side-by-side
    widths, heights = zip(*(i.size for i in imgs))
    total_width = sum(widths)
    max_height = max(heights)
    new_im = Image.new('RGB', (total_width, max_height))
    x_offset = 0
    for im in imgs:
        new_im.paste(im, (x_offset,0))
        x_offset += im.size[0]
    display(new_im)

# Example usage:
# show_before_after('Mixa Cream')   # expects data/images/Mixa Cream/before.jpg and after.jpg
